# Klasifikasi ECG PhysioNet Challenge 2017 dengan LSTM

Langkah-langkah:
1. Pre-processing: filter baseline, normalisasi
2. Sliding window / segmentasi
3. R-peak detection / beat extraction
4. Training LSTM untuk klasifikasi

In [ ]:
# Import library
import os
import numpy as np
import pandas as pd
import wfdb
import neurokit2 as nk
from scipy import signal
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Pre-processing function
def preprocess_ecg(raw_signal, fs=300):
    nyquist = fs / 2
    b, a = signal.butter(4, [0.5/nyquist, 40/nyquist], btype='band')
    filtered = signal.filtfilt(b, a, raw_signal)
    scaler = MinMaxScaler()
    normalized = scaler.fit_transform(filtered.reshape(-1, 1)).flatten()
    return normalized

# Sliding window segmentation
def segment_signal(signal, window_size, step):
    segments = []
    for start in range(0, len(signal) - window_size, step):
        segments.append(signal[start:start+window_size])
    return np.array(segments)

# R-peak detection
def get_r_peaks(ecg, fs):
    try:
        _, info = nk.ecg_process(ecg, sampling_rate=fs)
        return info['ECG_R_Peaks']
    except:
        return []

# Load label mapping
label_map = {'N': 0, 'A': 1, 'O': 2, '~': 3}
label_names = ['Normal', 'AF', 'Other', 'Noisy']

# Load reference labels
reference_df = pd.read_csv('dataset/REFERENCE-v3.csv', header=None, names=['record', 'label'])
print(f"Total records: {len(reference_df)}")
print(f"Label distribution:\n{reference_df['label'].value_counts()}")

# Process multiple records
X_all = []
y_all = []

# Process first 100 records as example (ubah jadi lebih banyak untuk training full)
num_records = 100
window_size = 600  # 2 detik pada 300Hz
step = 300  # 1 detik overlap

print(f"\nProcessing {num_records} records...")
for idx, row in reference_df.head(num_records).iterrows():
    record_id = row['record']
    label = row['label']
    
    try:
        # Load record
        record_path = f'dataset/training2017/{record_id}'
        record = wfdb.rdrecord(record_path)
        fs = record.fs
        ecg = record.p_signal[:,0]
        
        # Pre-process
        ecg_clean = preprocess_ecg(ecg, fs)
        
        # Segmentasi
        segments = segment_signal(ecg_clean, window_size, step)
        
        # Add segments and labels
        X_all.extend(segments)
        y_all.extend([label_map[label]] * len(segments))
        
        if (idx + 1) % 20 == 0:
            print(f"Processed {idx + 1}/{num_records} records...")
            
    except Exception as e:
        print(f"Error processing {record_id}: {e}")
        continue

# Convert to numpy arrays
X_all = np.array(X_all)
y_all = np.array(y_all)

print(f"\nTotal segments: {len(X_all)}")
print(f"X shape: {X_all.shape}")
print(f"Label distribution in segments: {np.bincount(y_all)}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# Reshape for LSTM
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

# Convert labels to categorical
y_train_cat = to_categorical(y_train, num_classes=4)
y_test_cat = to_categorical(y_test, num_classes=4)

print(f"\nTrain set: {X_train.shape}, Test set: {X_test.shape}")

# Build improved model
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(window_size, 1)),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(4, activation='softmax')  # 4-class classification
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Training
history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=20,
    batch_size=32,
    verbose=1
)

# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test_cat)
print(f"\nTest Accuracy: {test_acc:.4f}")

# Save model
model.save('model/keras_model_1.h5')
print("Model saved to model/keras_model_1.h5")

Total records: 8528
Label distribution:
label
N    5076
O    2415
A     758
~     279
Name: count, dtype: int64

Processing 100 records...
Processed 20/100 records...
Processed 40/100 records...
Processed 60/100 records...
Processed 80/100 records...
Processed 100/100 records...

Total segments: 3269
X shape: (3269, 600)
Label distribution in segments: [1903  301  981   84]

Train set: (2615, 600, 1), Test set: (654, 600, 1)


/home/bintang/Documents/GitHub/adikaraaa/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 600, 128)       │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 600, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,180 (461.64 KB)

 Trainable params: 118,180 (461.64 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 84s 948ms/step - accuracy: 0.5797 - loss: 1.0240 - val_accuracy: 0.5826 - val_loss: 0.9625
Epoch 2/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 77s 939ms/step - accuracy: 0.5816 - loss: 0.9903 - val_accuracy: 0.5826 - val_loss: 0.9672
Epoch 3/20
50/82 ━━━━━━━━━━━━━━━━━━━━ 31s 976ms/step - accuracy: 0.5954 - loss: 0.9674